# ML Features Quick-Start: Predict Trade Outcomes

This notebook shows how to load the `ml_features.parquet` file produced by the backtester and train a basic scikit-learn classifier to predict `is_win` (whether a trade is profitable).

## Prerequisites

1. Run a backtest with `export_ml_features: True` in `config.py`
2. Install scikit-learn: `pip install scikit-learn`
3. Update the `PARQUET_PATH` below to point to your run's output file

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- UPDATE THIS PATH to your actual run output ---
PARQUET_PATH = Path("../output/runs/REPLACE_WITH_YOUR_RUN_ID/ml_features.parquet")
# Load the data
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df):,} trades across {df['Strategy'].nunique()} strategies")
print(f"Win rate: {df['is_win'].mean():.1%}")
df.head()

## Explore the Feature Columns

The `entry_*` columns capture the market state at the moment each trade was entered. These are the features we'll use for prediction.

In [ ]:
# Identify the entry feature columns
feature_cols = [c for c in df.columns if c.startswith("entry_")]
print(f"Found {len(feature_cols)} entry features:")
for col in feature_cols:
    non_null = df[col].notna().sum()
    print(f"  {col:30s}  non-null: {non_null:>6,} / {len(df):,}")

In [ ]:
# Quick correlation of features with the target
target = "is_win"
corr = df[feature_cols + [target]].corr()[target].drop(target).sort_values()
print("Feature correlations with is_win:")
print(corr.to_string())

## Train a Basic Classifier

I'll use a Random Forest as a quick baseline. The goal isn't to build a production model — it's to show the workflow and see which features matter.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Prepare the dataset
# Drop rows where any feature is NaN (indicator warm-up period)
ml_df = df[feature_cols + [target]].dropna()
print(f"Rows after dropping NaN: {len(ml_df):,} (dropped {len(df) - len(ml_df):,})")

X = ml_df[feature_cols]
y = ml_df[target]

# Train/test split — use a time-aware split in production!
# Here we use a random split for simplicity.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Train win rate: {y_train.mean():.1%}  Test win rate: {y_test.mean():.1%}")

In [ ]:
# Train a Random Forest
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,           # Keep shallow to reduce overfitting
    min_samples_leaf=20,   # Require at least 20 trades per leaf
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.3f}")

In [ ]:
# Feature importance
importance = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top features by importance:")
print(importance.to_string())

In [ ]:
import matplotlib.pyplot as plt
importance.plot(kind="barh")
plt.xlabel("Importance")
plt.title("Feature Importance for Predicting Trade Outcomes")
plt.tight_layout()

## Interpreting the Results

**ROC AUC close to 0.50** means the entry features alone don't predict winners vs losers — which is expected for a well-designed trend-following strategy (the edge comes from position sizing and cutting losses, not from entry timing).

**ROC AUC above 0.55** suggests some entry features carry predictive signal. Look at which features rank highest in importance — they might reveal regime dependencies (e.g., VIX level at entry, RSI at entry).

**Next steps:**
- Add `HoldDuration`, `RMultiple` as features (non-entry context)
- Try a time-series split instead of random split
- Group by strategy and train separate models per strategy
- Use the predictions to filter which trades to actually take in a live system